![Platform+](images/banner.png)

# Lab 9: Unstructured Data + AI
## Coffee Menu Intelligence — make PDFs and images queryable with AI.

> **Persona track: Data Analyst** — same lab, your angle.

:point_right: **In this lesson, you'll turn a collection of coffee knowledge-base PDFs into a searchable system that can answer any menu question instantly.**

The scenario: Your coffee shop has a kiosk on the counter. Customers can ask natural-language questions about drinks. Behind the scenes, AI parses PDF documents and answers questions.

**First — what is "unstructured data"?** Most analytics runs on *structured* data: rows and columns in tables. But the majority of enterprise data isn't like that — it's **unstructured**: PDFs, images, audio, video, emails, and free text that don't fit neatly into columns. Historically it sat in storage untouched because SQL couldn't read it. That's the shift in this lab: Snowflake's AISQL functions turn unstructured files into structured, queryable data — so the SQL skills you already have now reach the roughly 80% of enterprise data that used to be off-limits.

We'll cover:
1. Explore PDF files in a Snowflake stage
2. Parse documents with `AI_PARSE_DOCUMENT`
3. Extract structured fields with `AI_EXTRACT`
4. Build a knowledge-base table from all PDFs
5. Query the knowledge base with `AI_COMPLETE`
6. Analyze coffee images with multimodal AI

📚 **By the end you'll be able to:**
- Parse PDFs into queryable text using AI_PARSE_DOCUMENT
- Extract structured fields from unstructured documents
- Build an AI-powered Q&A system from a document collection
- Use multimodal AI to analyze images

**Time:** ~45 min · **Lab interface:** Cortex Code (CoCo) is your tutor — ask it to explain or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; anything you create lives in your own database (named after your username, e.g. `USER_DB`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set your session context

Every Snowflake Notebook needs a session context: which role, warehouse, database, and schema you're working in. This is the same pattern you'd use in any production script.

> :bulb: **Real-world note:** In production, context is typically set by the application or service account — not manually. But in a lab, setting it explicitly ensures everyone starts from the same place.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_RAW;


In [ ]:
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RAW;

## 1 · Explore the stage

In Snowflake, **stages** are where files live — PDFs, images, CSVs, videos. They're like cloud storage (S3/Azure Blob) but managed within Snowflake's security perimeter.

The roastery uploaded one PDF per coffee drink into `@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE/coffee_kb/`. Let's see what's there.

> :bulb: **Real-world context:** In production, files arrive in stages via automated pipelines (Snowpipe, PUT commands, cloud event triggers). The stage is your "inbox" for unstructured data. Everything downstream starts here.

In [ ]:
LIST @PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE/coffee_kb/;

### Directory table view

A [**directory table**](https://docs.snowflake.com/en/user-guide/data-load-dirtables) isn't a separate database object — it's an *implicit metadata layer* that any Snowflake stage can expose (enabled with `DIRECTORY = (ENABLE = TRUE)` on the stage). Once it's on, instead of the raw `LIST` command you query the stage through the `DIRECTORY()` **table function**, which returns one row per file with columns like `RELATIVE_PATH`, `SIZE`, `LAST_MODIFIED`, `MD5`, `ETAG`, and `FILE_URL`.

Because it behaves like a normal table, you get the full power of SQL over your files — `WHERE`, `JOIN`, `GROUP BY`, `ORDER BY`. Think of it as a queryable file-system inventory that lives inside your SQL engine.

> :bulb: **`LIST` vs `DIRECTORY()`:** `LIST` is a quick command that prints file names to the results grid, but you can't filter, join, or build on its output. `DIRECTORY()` returns a true result set you can query, wrap in a view, or feed into a pipeline — which is exactly what we do in Section 4.

In [ ]:
SELECT RELATIVE_PATH, SIZE, LAST_MODIFIED
FROM DIRECTORY('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE')
WHERE RELATIVE_PATH LIKE 'coffee_kb/%.pdf'
ORDER BY RELATIVE_PATH;

> :mag_right: **You should see 8 PDF files** — one per coffee drink (espresso, latte, cappuccino, americano, cold brew, flat white, mocha, pour over).
>
> **Use case:** In a real deployment, you might have thousands of files landing daily — invoices, reports, contracts. The DIRECTORY table lets you identify what's new (by LAST_MODIFIED), filter by naming convention (with LIKE/REGEXP), and process only what you need.

## 2 · Parse a single document

[`AI_PARSE_DOCUMENT`](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document) is the function that extracts the content of a binary file (PDF, image, scan) and returns it as structured JSON — the text lives in the `content` field, which your SQL engine can then work with. It's the bridge between unstructured and structured worlds.

**Two modes:**
| Mode | Best For | Output |
|------|----------|--------|
| **OCR** | Scanned images, handwritten notes, photos of text | Raw text, no structure preserved |
| **LAYOUT** | Formatted documents (reports, forms, PDFs with headings) | Structured text preserving headings, tables, paragraphs |

We'll use **LAYOUT** mode because our coffee PDFs have clear section headings (History, Ingredients, Preparation) that we want preserved for downstream extraction.

> :bulb: **Best practice:** Always start with LAYOUT mode. Only fall back to OCR if your documents are scanned images or if LAYOUT produces garbled output on your specific file format.

**What the cell below does:** We call `AI_PARSE_DOCUMENT` on a *single* file first — `espresso.pdf` — before scaling to the whole stage in Section 4. `TO_FILE('@...DOCS_STAGE', 'coffee_kb/espresso.pdf')` hands the staged binary to the function; `{'mode': 'LAYOUT'}` selects the parsing mode; and `:content::STRING` pulls the parsed text out of the object the function returns (it also returns `images` and `layout` metadata we don't need here). Starting with one known document is the right habit — validate parse quality on a file you understand before committing to a bulk run.

> 📌 **Timing varies.** AI function execution depends on current inference capacity — the same cell might complete in seconds or take several minutes depending on demand. If it's slow, let it run; it will complete.

In [ ]:
SELECT AI_PARSE_DOCUMENT(
  TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_kb/espresso.pdf'),
  {'mode': 'LAYOUT'}
):content::STRING AS parsed_text;

> :mag_right: **Review the output.** You should see structured text with headings (History, Ingredients, Preparation, etc.) extracted from the PDF. The section structure is preserved — this is what LAYOUT mode gives you.
>
> **Quality check:** Does the text match what's in the actual PDF? Are any sections missing or garbled? In production, you'd spot-check 5–10 documents before building on top. Parse quality = answer quality downstream.
>
> **Real-world difference:** Our lab PDFs are cleanly formatted. Real-world documents are messier — multi-column layouts, watermarks, headers/footers, scanned at angles. AI_PARSE_DOCUMENT handles most of these, but edge cases exist. Always validate.

<details>
<summary>👀 <b>Optional: open the source PDF and check the parse yourself</b> (click to expand)</summary>

Want to eyeball the original `espresso.pdf` against the parsed text above? Run the cell below — it downloads the PDF into the notebook's runtime and hands you a self-contained download link.

**Why download-and-embed instead of a presigned URL?** Inside a Workspace/SPCS runtime, `GET_PRESIGNED_URL` on an *internal* stage returns a link scoped to a storage access point your browser can't reach (you'd get an S3 `AccessDenied`). Pulling the bytes into the runtime with `session.file.get()` and building a `data:` URL sidesteps that entirely — the link carries the file itself, so no second trip to storage is needed.

| Function | What you get | Works from a plain browser? |
|---|---|---|
| `session.file.get()` + `data:` URL (used here) | The file pulled into the runtime, embedded in the link | ✅ Yes — self-contained |
| `GET_PRESIGNED_URL` | A temporary storage link | ⚠️ Not from a Workspace/SPCS runtime (access-point deny) |
| `BUILD_SCOPED_FILE_URL` | A short-lived scoped URL (used inside apps like the kiosk in Section 10) | ❌ Needs a Snowflake session |
| `BUILD_STAGE_FILE_URL` / `FILE_URL` | A permanent stage URL | ❌ Needs a Snowflake session |

</details>

In [ ]:
# Optional: download the source PDF into the runtime and build a self-contained link
import base64
from IPython.display import display, HTML

session.file.get('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE/coffee_kb/espresso.pdf', '/tmp/')

with open('/tmp/espresso.pdf', 'rb') as f:
    pdf_b64 = base64.b64encode(f.read()).decode()

display(HTML(
    f'<a href="data:application/pdf;base64,{pdf_b64}" download="espresso.pdf" target="_blank">'
    f'📄 Download / open espresso.pdf</a> '
    f'&nbsp;<span style="color:#888">(the original PDF, embedded in this link)</span>'
))

> 💡 **If a link doesn't open when you click it** (a known Snowsight quirk with links rendered inside a notebook), **right-click it → Open in new tab.** The same applies to the documentation links throughout this lab.

## 3 · Extract structured fields from parsed text

Parsing gives you raw text. But what if you need SPECIFIC fields — caffeine content, serving temperature, drink name — as clean columns you can filter and aggregate?

[`AI_EXTRACT`](https://docs.snowflake.com/en/sql-reference/functions/ai_extract) takes your parsed text and pulls out named fields. You define WHAT you want; the AI figures out WHERE it is in the text.

> :bulb: **You've met this before:** You first used `AI_EXTRACT` back in **Lab 5**, on plain text columns. It's the same function — here we simply point it at text we just parsed out of a PDF. Once a document is text, every AISQL function you already know works on it.

**The pattern:** `AI_PARSE_DOCUMENT` (get text) → `AI_EXTRACT` (get fields from text). These two functions chain together for any document processing workflow.

> :bulb: **Use case:** Extract invoice totals from supplier invoices. Extract clause types from legal contracts. Extract diagnosis codes from medical records. The function is the same — only your field names change.

In [ ]:
WITH doc AS (
  SELECT AI_PARSE_DOCUMENT(
    TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_kb/espresso.pdf'),
    {'mode': 'LAYOUT'}
  ):content::STRING AS text
)
SELECT
  AI_EXTRACT(text, ['drink_name', 'caffeine_mg', 'serving_temperature', 'preparation_time']) AS extracted
FROM doc;

> :mag_right: **Read the query in two parts:**
> - The `doc` [CTE](https://docs.snowflake.com/en/user-guide/queries-cte) calls `AI_PARSE_DOCUMENT` **once** to get the espresso PDF's text. (In Section 4 you'll parse all 8 files at once and store them; here we parse inline just to demo extraction.)
> - `AI_EXTRACT(text, ['drink_name', ...])` then works **on that already-parsed text** — it is *not* re-reading or re-parsing the PDF. You hand it a list of field names and it returns an object with a value for each.

The cell below doesn't do anything new — it runs the **same extraction query** you just saw, then uses a little Python (`json.loads` + a Markdown table) to render the result as a clean **field | value** table instead of a raw JSON blob. Same data, friendlier output.

In [ ]:
# Show the same extraction as a clean field | value table
import json
from IPython.display import display, Markdown

resp = session.sql("""
    WITH doc AS (
      SELECT AI_PARSE_DOCUMENT(
        TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_kb/espresso.pdf'),
        {'mode': 'LAYOUT'}
      ):content::STRING AS text
    )
    SELECT AI_EXTRACT(text, ['drink_name', 'caffeine_mg', 'serving_temperature', 'preparation_time']):response AS fields
    FROM doc
""").collect()[0][0]

fields = json.loads(resp) if isinstance(resp, str) else resp

table = '| Field | Extracted value |\n|---|---|\n'
for k, v in fields.items():
    table += f'| `{k}` | {v} |\n'
display(Markdown(table))

> :mag_right: **Check the extraction.** Did AI_EXTRACT correctly identify the drink name, caffeine content, and serving temperature from the unstructured text?
>
> **Things to watch for:**
> - Did it return just the number ("63") or the full phrase ("63 mg per single shot")? Consistency matters for dashboards.
> - Did it find all fields, or return null for some? Null means the AI couldn't locate that information in the text.
> - Is the extracted value CORRECT when you compare it to the source PDF?
>
> **Real-world tip:** If extraction is inconsistent, make your field names more specific: `caffeine_mg_number` instead of `caffeine_mg`. The AI takes your field name as a hint for what format to return.

## 4 · Build the knowledge-base table

Now the production pattern: parse ALL documents at once and store the results in a table. This is your **knowledge base** — the foundation for search, Q&A, and the customer kiosk.

**The key insight:** You're combining two Snowflake features:
- `DIRECTORY()` — iterates over all files in a stage (like a file listing)
- `AI_PARSE_DOCUMENT()` — converts each file to text

Together, they create a scalable document ingestion pipeline in a single SQL statement.

> :bulb: **Real-world scale:** In production, this same pattern processes thousands of documents. A financial services firm might parse 50,000 quarterly filings. A legal team might parse 10,000 contracts. The SQL is identical — only the stage path and the table name change.

> 📌 **Timing varies.** AI function execution depends on current inference capacity — the same cell might complete in seconds or take several minutes depending on demand. If it's slow, let it run; it will complete.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE AS
SELECT
  SPLIT_PART(RELATIVE_PATH, '/', -1) AS source_file,
  REPLACE(SPLIT_PART(RELATIVE_PATH, '/', -1), '.pdf', '') AS drink_id,
  AI_PARSE_DOCUMENT(
    TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', RELATIVE_PATH),
    {'mode': 'LAYOUT'}
  ):content::STRING AS content
FROM DIRECTORY('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE')
WHERE RELATIVE_PATH LIKE 'coffee_kb/%.pdf';

In [ ]:
-- Verify: how many drinks in our knowledge base?
SELECT drink_id, LENGTH(content) AS content_length
FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
ORDER BY drink_id;

> :bulb: **What just happened:** In one SQL statement, you converted 8 raw PDF files into 8 rows of queryable text. Each row has a `drink_id` (derived from the filename) and the full `content` (parsed text).
>
> This table is now your **single source of truth** for coffee knowledge. Every question the kiosk answers comes from this data. That's why parse quality matters — if a PDF wasn't parsed correctly, the system can't answer questions about that drink.
>
> **Best practice:** In production, add a `parsed_at` timestamp column and track which files have been processed. This enables incremental refresh: only parse NEW files on each run.

## 5 · Ask questions of your knowledge base

Here's where it gets powerful. You have text in a table. Now use [`AI_COMPLETE`](https://docs.snowflake.com/en/sql-reference/functions/ai_complete) to answer natural-language questions **grounded in your documents**.

> :bulb: **About `AI_COMPLETE`:** This is the general-purpose text-generation function in Cortex — you pass it a **model name** and a **prompt**, and it returns the model's response. The first argument lets you swap models freely (`llama3.1-70b`, `mistral-large2`, `claude-*`, and more — see the [AISQL function overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql) for the current model list). We use **`llama3.1-70b`** here as a balanced default: large enough to synthesise facts across all 8 documents and follow the "answer only from this context" instruction, but far cheaper than the top-tier models (and unlike the 8b, it won't fumble multi-fact answers). Picking the right model for the job — cost vs. quality — is covered in depth in the **Admin** notebook.

**The pattern:**
1. Retrieve knowledge ([LISTAGG](https://docs.snowflake.com/en/sql-reference/functions/listagg) all content into one string)
2. Inject it as context into a prompt
3. Ask your question
4. The LLM answers using ONLY your provided context

This is called [**Retrieval-Augmented Generation (RAG)**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview) — the most common pattern in enterprise AI today. The LLM doesn't "know" about your coffee menu from training. It reads your documents at query time and answers from them.

> :warning: **Limitation:** LISTAGG concatenates ALL documents into one prompt. This works for our 8 small PDFs (~15,000 tokens). At 100+ documents or larger files, you'd hit token limits and need **Cortex Search** instead (the Data Engineer notebook builds this).

In [ ]:
-- 👇 EDIT YOUR QUESTION HERE — everything below just injects it into the prompt
SET question = 'Which coffee drink has the highest caffeine content and why?';

-- Aggregate all knowledge and ask the question set above
SELECT AI_COMPLETE(
  'llama3.1-70b',
  'You are a coffee expert. Based on the following knowledge base, answer the question.\n\n'
  || 'KNOWLEDGE BASE:\n' || (
    SELECT LISTAGG(content, '\n---\n') FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
  )
  || '\n\nQUESTION: ' || $question
) AS answer;

In [ ]:
-- 👇 EDIT YOUR QUESTION HERE — try any menu question you like
SET question = 'What is the difference between a flat white and a latte?';

SELECT AI_COMPLETE(
  'llama3.1-70b',
  'You are a coffee expert. Based on the following knowledge base, answer the question.\n\n'
  || 'KNOWLEDGE BASE:\n' || (
    SELECT LISTAGG(content, '\n---\n') FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
  )
  || '\n\nQUESTION: ' || $question
) AS answer;

> :mag_right: **Quality check.** Look at the answer carefully:
> - Does it cite **specific numbers** ("200-250mg", "63mg") from your PDFs? That's grounded.
> - Or does it give **generic information** ("espresso-based drinks tend to have more caffeine")? That might be the model's training data, not your knowledge base.
>
> **The grounding test:** If the answer includes a fact that's ONLY in your PDFs and not general knowledge, it's grounded. If you can't tell whether it came from your docs or the internet, that's a signal to improve your prompting (e.g., add "Only use information from the KNOWLEDGE BASE above. If you don't know, say so.").
>
> **Real-world context:** This grounding verification is critical before deploying to customers. An agent that confidently gives wrong information is worse than one that says "I don't know." Always validate before go-live.

## 6 · Build a reusable Q&A function

Let's package the Q&A pattern into a reusable **SQL function** (a UDF). This is the same logic you'd put behind a Streamlit app, a REST API, or any customer-facing interface — but as a database object any tool can call with plain SQL.

**Why a function?** Because the LISTAGG + AI_COMPLETE pattern is always the same — only the question changes. Wrapping it in a UDF means you (or CoCo) can call it from any UI, dashboard, or query without rewriting the SQL. Crucially, the question is **not hard-coded** — you pass it in as an argument, so the same function answers *any* question you ask. That flexibility is the whole point: one function, unlimited questions.

> :bulb: **Production note:** In a real deployment, you wouldn't LISTAGG every time. You'd either:
> 1. Cache the knowledge base content (fetch once, reuse for multiple questions)
> 2. Use **Cortex Search** to retrieve only relevant documents per question (the scalable approach)
>
> The UDF below is the prototype. The Data Engineer notebook shows the production-grade version with Cortex Search + Agent.

In [ ]:
-- Package the Q&A pattern into a reusable SQL function (UDF).
-- The ::STRING cast is required — AI_COMPLETE returns VARIANT otherwise.
CREATE OR REPLACE FUNCTION {{my_db}}.RAW.ASK_COFFEE_EXPERT(question STRING)
RETURNS STRING
AS
$$
  AI_COMPLETE(
    'llama3.1-70b',
    'You are a friendly coffee expert at a roastery. Based on the knowledge base below, answer concisely.\n\n'
    || 'KNOWLEDGE BASE:\n'
    || (SELECT LISTAGG(content, '\n---\n') FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE)
    || '\n\nQUESTION: ' || question
  )::STRING
$$;

-- Try it out — change the question here:
SELECT {{my_db}}.RAW.ASK_COFFEE_EXPERT(
  'What is the best drink for someone who does not like strong coffee?'
) AS answer;

### Your turn — ask your own question

The `ASK_COFFEE_EXPERT` function now exists in your database. Edit the question text in the cell below and run it to ask **anything** about the coffee menu.

In [ ]:
-- 👇 YOUR TURN — edit the question text, then run this cell.
-- Stuck for ideas? Try one of these:
--   • 'Which drink is best if I want the least caffeine?'
--   • 'What should I order if I do not like bitter coffee?'
--   • 'Compare cold brew and a latte for someone new to coffee.'
SELECT {{my_db}}.RAW.ASK_COFFEE_EXPERT(
  'Which drink has the most caffeine, and which has the least?'
) AS answer;

> :bulb: **What you built:** A reusable Q&A **SQL function (UDF)** backed by your parsed PDF knowledge base. The key insight:
>
> | Component | Role |
> |-----------|------|
> | `LISTAGG(content)` | Retrieves all knowledge |
> | The prompt template | Tells the AI its role and constraints |
> | `AI_COMPLETE` | Generates the answer |
> | `$$...$$` (dollar-quoting) | Safely passes the prompt without escaping issues |
>
> **Real-world scaling question:** What happens when you have 10,000 documents instead of 8? LISTAGG would produce a string too large for any model's context window. That's why production systems use **Cortex Search** to retrieve only the 3–5 most relevant documents per question. Same pattern, smarter retrieval.

---
## :robot_face: CoCo Assist

Your store manager asks: *"Can you compare two drinks side by side? I want to help customers decide between options."*

Instead of writing this yourself, let's use **Cortex Code (CoCo)** to generate the SQL. This is the real-world workflow: you know WHAT you want, CoCo handles the HOW.

**What this prompt asks CoCo to do (in plain language):** Get the AI to return its comparison as structured data (JSON) instead of free text — then convert that structured data into a clean table with one row per comparison category. The result is a neat 3-row table comparing espresso vs latte across caffeine, preparation, and flavor.

**Jargon decoder:**
| Term you'll encounter | What it means |
|---|---|
| `AI_COMPLETE` | The Snowflake function that sends a prompt to an AI model and gets a response |
| `LISTAGG` | Combines all rows from a column into one big string — here it merges all coffee documents into one block of context |

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/sql-author You are a coffee expert. Using ONLY the data in the @{{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE table, write a SQL query that compares espresso and latte. I would like the result to be a table that provides a concise side-by-side comparison of the two drinks across three categories: caffeine, preparation method, and flavour profile. Use the AI SQL functions, such as AI_COMPLETE, to assist with the query. Test the SQL to ensure it is correct and returns rows. Do not add the final SQL as a cell to the notebook.</pre>

**What to do with the result:**
> 1. Copy CoCo's generated SQL from the CoCo panel
> 2. Paste it into the empty cell below
> 3. Run it
> 4. If it errors, paste the error message back to CoCo — it will fix the query and you'll see **Keep** / **Undo** buttons
>
> 💡 If CoCo adds the SQL as a new cell at the bottom of the notebook anyway, just copy it from there into the cell below and delete the extra cell.

This iterate-with-CoCo workflow is how analysts build SQL without memorizing syntax.

In [ ]:
-- Paste CoCo's suggestion on the blank line below, then run it


<details>
<summary>✅ <b>Reference solution</b> — reveal only if you're stuck or want to compare (click to expand)</summary>

CoCo's exact output will vary (LLMs are non-deterministic), but here's one working version you can copy into the empty cell above and run:

```sql
WITH coffee_context AS (
  SELECT LISTAGG(CONTENT, '\n---\n') WITHIN GROUP (ORDER BY DRINK_ID) AS knowledge
  FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE
  WHERE DRINK_ID IN ('espresso', 'latte')
),
ai AS (
  SELECT AI_COMPLETE(
    'llama3.1-70b',
    'You are a coffee expert. Using ONLY the following knowledge base, compare espresso and latte. '
    || 'Return ONLY a JSON array with exactly 3 objects, no other text. '
    || 'Each object must have keys: category, espresso, latte. '
    || 'The 3 categories are: Caffeine, Preparation Method, Flavour Profile.\n\n'
    || 'KNOWLEDGE BASE:\n' || knowledge
  ) AS raw_response
  FROM coffee_context
)
SELECT
  f.value:category::STRING AS category,
  f.value:espresso::STRING AS espresso,
  f.value:latte::STRING    AS latte
FROM ai,
-- REGEXP_SUBSTR pulls just the JSON array out, in case the model adds any stray text
LATERAL FLATTEN(input => PARSE_JSON(REGEXP_SUBSTR(raw_response, '\\[.*\\]', 1, 1, 's'))) AS f;
```

</details>

## 7 · Multimodal: Analyze coffee images with AI

So far we've worked with text (PDFs). But Cortex AI can also **see** — analyze images, describe what's in them, and answer questions about visual content.

**How it works:** The same `AI_COMPLETE` function you used for text Q&A also accepts image files via `TO_FILE()`. The model (`pixtral-large`) is multimodal — it processes both text prompts and visual inputs together.

**Why this matters for analysts:**
- Product photography analysis ("Does this match our brand standards?")
- Visual quality control ("Is this latte art acceptable for serving?")
- Content categorization ("What drink is in this customer's Instagram post?")
- Accessibility (generate alt-text descriptions for menu images)

---

### What happens in the cell below

The cell below creates an **interactive dropdown widget** (`ipywidgets.Dropdown`). When you select a drink from the dropdown:

1. The corresponding coffee image loads and displays below the dropdown.
2. A **Python string variable called `drink`** is set to the selected value (e.g. `'cappuccino'`, `'cold_brew'`).

This `drink` variable is a **global Python variable** — it persists in the notebook session and is available to all cells that run after it. The next cell (AI analysis) uses `drink` inside an f-string to build a SQL query that passes the correct image file to `AI_COMPLETE`.

**In short:** Pick a drink → variable is set → next cell uses that variable to analyze the image with AI. No typing required.

**👉 Run the cell below, select a drink from the dropdown, then run the AI analysis cell.**

In [ ]:
import ipywidgets as widgets
import requests
from IPython.display import display

# --- Load all coffee image bytes from the Snowflake stage ---
DRINKS = ['latte', 'espresso', 'cappuccino', 'americano', 'cold_brew', 'flat_white', 'mocha', 'pour_over']

image_bytes = {}
for d in DRINKS:
    url = session.sql(
        f"SELECT GET_PRESIGNED_URL('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_images/{d}.jpg')"
    ).collect()[0][0]
    image_bytes[d] = requests.get(url).content

# --- Dropdown widget ---
drink_selector = widgets.Dropdown(
    options=[(d.replace('_', ' ').title(), d) for d in DRINKS],
    value='latte',
    description='\u2615 Drink:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='280px')
)

# --- Image widget (updates in-place when dropdown changes) ---
image_widget = widgets.Image(
    value=image_bytes['latte'],
    format='jpg',
    width=400
)

# --- Label showing current selection ---
label = widgets.HTML(value='<h3>Latte</h3>')

def on_drink_change(change):
    """Callback: update the global `drink` variable and swap the image."""
    global drink
    drink = change['new']
    image_widget.value = image_bytes[drink]
    label.value = f'<h3>{drink.replace("_", " ").title()}</h3>'

drink_selector.observe(on_drink_change, names='value')

# Set initial variable
drink = drink_selector.value

# Display everything as a vertical layout
display(widgets.VBox([drink_selector, label, image_widget]))

Now pass your selected drink image directly to `AI_COMPLETE` using [`TO_FILE()`](https://docs.snowflake.com/en/sql-reference/functions/to_file). `TO_FILE()` builds a lightweight reference to a file sitting on a stage — it doesn't copy the bytes, it just hands the AI function a pointer it can read. That's how the same `AI_COMPLETE` you used for text can now "see" an image. The `pixtral-large` model is multimodal — it reads the image and the text prompt together, then returns a natural-language description covering drink type, visual characteristics, and preparation quality.

Note that the SQL function is enclosed in a Python wrapper. This is to improve readibility of the output.



In [ ]:
from IPython.display import display, Markdown

result = session.sql(f"""
    SELECT AI_COMPLETE(
      'pixtral-large',
      'Describe this coffee drink image in detail. What drink is it? What can you tell about the preparation quality?',
      TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', lower('coffee_images/{drink}.jpg'))
    ) AS image_description
""").collect()[0][0]

result_clean = result.strip('"').replace('\\n', '\n')
display(Markdown(f'**{drink.replace("_", " ").title()} — AI Analysis**\n\n{result_clean}'))

> :point_right: **Try it yourself:** scroll back up to the **drink dropdown**, pick a *different* drink, then re-run the analysis cell above. Watch how the AI's description changes to match the new image — same query, different input. Do a couple to get a feel for how consistent (or not) the model is across drinks.

> :mag_right: **Multimodal AI** can see images AND read text in the same call. Think about what this enables:
>
> - A customer photographs a drink on another table → AI identifies it → provides info from the KB
> - QA team photographs each drink before serving → AI rates presentation quality
> - Social media monitoring → AI identifies which of your products appears in customer posts
>
> **Real-world difference:** Our lab images are clean product photos. Real customer photos might be poorly lit, at odd angles, or include background clutter. Production multimodal systems need to handle this gracefully — either by asking for a better photo or providing a confidence score with the identification.

> :bulb: **The multimodal progression:** In this lab you've worked with PDFs and images so far. Two more input types — video and audio — are covered in the next sections. Here's the full picture of what Cortex AI can process:
>
> | Input type | Function | Model | Example |
> |-----------|----------|-------|---------|
> | PDF/Documents | `AI_PARSE_DOCUMENT` | Built-in OCR/Layout | Extract text from coffee knowledge PDFs |
> | Images | `AI_COMPLETE` + `TO_FILE` | `pixtral-large` | Describe latte art quality |
> | Video | `AI_COMPLETE` + `TO_FILE` | `gemini-3.5-flash` | Analyze barista preparation technique |
> | Audio | `AI_COMPLETE` + `TO_FILE` | `gemini-3.5-flash` | Transcribe and analyze customer calls |
>
> ⚠️ Video and audio analysis via `AI_COMPLETE` are currently in **Public Preview**. `AI_TRANSCRIBE` for audio transcription is **GA**.
>
> The pattern is always the same: stage the file → reference it with `TO_FILE` → ask AI a question. The model choice determines which media types are supported.

---
## Verify Your Work

Before continuing to the video and audio sections, confirm your core pipeline is working:

| Check | Expected Result |
|-------|----------------|
| Stage listing | 8 PDF files in `coffee_kb/` |
| Parsed text | Structured content with clear headings |
| Knowledge base table | 8 rows, all with non-null content |
| AI Q&A | Answers cite specific details from your PDFs |
| Image analysis | AI correctly identifies the drink type |

Run the check below:

In [ ]:
SELECT COUNT(*) AS kb_rows FROM {{my_db}}.RAW.COFFEE_KNOWLEDGE_BASE;

## 8 · Video analysis with Gemini

Snowflake Cortex now supports **video and audio analysis** using multimodal models. This means you can analyze video files staged in Snowflake the same way you analyze images — ask questions about what's happening, extract insights, and detect patterns.

**Why this matters for Data Analysts:**
- **Customer insights** — Analyze video testimonials, social media clips, or in-store footage at scale
- **Content intelligence** — Summarize video reviews without watching them manually
- **Quality assurance** — Verify barista preparation techniques match training standards
- **Brand monitoring** — Detect brand mentions and sentiment in video content

The syntax is identical to image analysis: `AI_COMPLETE` + `TO_FILE` pointing to a video file. The model choice (`gemini-3.5-flash`) determines multimodal capability.

> :warning: **Production note:** Video analysis uses significantly more tokens than text or image analysis due to frame-by-frame processing. Always estimate cost with `AI_COUNT_TOKENS` before running on large video collections. Consider extracting key frames or using shorter clips to manage costs.

### Play the video first

Before we ask AI anything about it, let's watch the video ourselves so we know what to expect. The cell below downloads the video file and renders it inline for playback directly in the notebook.

In [ ]:
# Generate a playable link for the coffee video
from IPython.display import display, HTML
import base64

session.file.get('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE/coffee_videos/coffee_making_and_review.mp4', '/tmp/')

with open('/tmp/coffee_making_and_review.mp4', 'rb') as f:
    video_bytes = f.read()

video_b64 = base64.b64encode(video_bytes).decode()
html = (
    f'<h3>Coffee Video</h3>'
    f'<video width="200" controls>'
    f'<source src="data:video/mp4;base64,{video_b64}" type="video/mp4">'
    f'</video>'
)
display(HTML(html))

> :mag_right: **Watch the video.** You should see someone making a coffee drink and giving a spoken review. Pay attention to what's happening AND what the reviewer says — in a moment we'll have AI describe the video (Section 8), then transcribe the audio (Section 9).

Now let's have AI describe the video for us. The cell below sends the video file to a multimodal model and asks what's happening. After it runs, compare the AI's description against what you just watched — did it correctly identify the drink and the steps? Did it catch anything you missed, or miss anything you noticed?

In [ ]:
-- Analyze a coffee preparation video
SELECT AI_COMPLETE(
  'gemini-3.5-flash',
   'Describe what is happening in this video. What coffee drink is being prepared? Rate the barista technique.',
   TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_videos/coffee_making_and_review.mp4')
 ) AS video_analysis;

## 9 · Transcribe the video audio with AI_TRANSCRIBE

The video has a spoken component — a customer giving a verbal review. [`AI_TRANSCRIBE`](https://docs.snowflake.com/en/sql-reference/functions/ai_transcribe) converts speech to text, turning audio into data you can analyze with all the text AI functions you already know.

**Why this matters for analysts:**
- **Voice of Customer** — Transcribe customer calls, video reviews, voice messages at scale
- **Meeting intelligence** — Convert recorded meetings to searchable text
- **Accessibility** — Generate captions and transcripts for video content
- **Chain with other AI functions** — Once transcribed, run AI_SENTIMENT, AI_CLASSIFY, AI_EXTRACT on the text

Let's try transcribing the coffee video.

In [ ]:
-- Attempt to transcribe the spoken audio from the coffee video
SELECT AI_TRANSCRIBE(
  TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_videos/coffee_making_and_review.mp4')
) AS transcription;

Most of you will see the transcript appear above. **If yours came back empty or null, don't worry** — expand the box below for the fix. Either way, the sentiment/classification cell further down re-transcribes a properly-encoded copy of the video for you, so you can keep going regardless.

<details>
<summary>🔧 <b>Got an empty or null transcript? Expand for the audio-codec fix</b> (click to expand)</summary>

**What happened:** the video file's audio track is encoded in a format `AI_TRANSCRIBE` can't process. Not all audio codecs are supported — the function needs a compatible encoding, and AAC (the default in most phone/screen-recording MP4s) is *not* one of them.

**How to diagnose it:** `AI_TRANSCRIBE` has a debug mode. Pass `TRUE` as the third argument to get a detailed error instead of a silent null. Paste this into a SQL cell and run it:

```sql
-- Debug mode: see WHY the transcription failed
SELECT AI_TRANSCRIBE(
  TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_videos/coffee_making_and_review.mp4'),
  {},
  TRUE
) AS debug_output;
```

You should see an error indicating the audio codec is unsupported. `AI_TRANSCRIBE` requires audio in one of: **FLAC, MP3, OPUS, VORBIS, or WAV**.

**The fix:** re-encode the audio track to a supported codec with ffmpeg:

```
ffmpeg -i coffee_making_and_review.mp4 -c:v copy -c:a libmp3lame -q:a 2 coffee_making_and_review_codec.mp4
```

This copies the video stream unchanged (`-c:v copy`) and re-encodes only the audio to MP3 (`-c:a libmp3lame`). A properly-encoded version has already been prepared and staged for you as `coffee_making_and_review_audio_encoding.mp4`. Transcribe that one:

```sql
-- Transcribe the properly-encoded video
SELECT AI_TRANSCRIBE(
  TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_videos/coffee_making_and_review_audio_encoding.mp4')
) AS transcription;
```

You should now see the spoken words converted to text — the customer's verbal review.

**The debugging pattern you just learned:**
1. Run the function → get a null/empty result
2. Re-run with debug mode (`TRUE`) → see the actual error
3. Fix the root cause (re-encode the audio)
4. Re-run on the fixed file → success

This applies to any AI function that returns unexpected results: check whether the INPUT is in the right format before assuming the function is broken.

</details>

### Analyze the transcription with AI

Now that you have text from the spoken review, chain it with two functions you already know:
- **`AI_SENTIMENT`** — scores the emotional tone of the text (positive / negative / neutral / mixed).
- **`AI_CLASSIFY`** — sorts the text into one label from a list you define (here: Positive Experience, Negative Experience, Mixed Feelings, General Commentary).

Both run directly on the transcribed text — no extra parsing needed:

In [ ]:
-- Score the sentiment and classify the spoken review
WITH transcript AS (
  SELECT AI_TRANSCRIBE(
    TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_videos/coffee_making_and_review_audio_encoding.mp4')
  ):text::STRING AS spoken_text
)
SELECT
  spoken_text,
  AI_SENTIMENT(spoken_text):categories[0].sentiment::STRING AS review_sentiment,
  AI_CLASSIFY(spoken_text, ['Positive Experience', 'Negative Experience', 'Mixed Feelings', 'General Commentary']):labels[0]::STRING AS review_type
FROM transcript
WHERE spoken_text IS NOT NULL;

> :bulb: **The multimodal insight:** You now have THREE complementary angles on one piece of content:
>
> | Analysis | Function | What It Tells You |
> |----------|----------|-------------------|
> | Visual analysis (Section 8) | `AI_COMPLETE` + gemini | What HAPPENED in the video |
> | Transcription (this section) | `AI_TRANSCRIBE` | What was SAID |
> | Sentiment + classification | `AI_SENTIMENT` + `AI_CLASSIFY` | How the reviewer FELT |
>
> This combined analysis is far richer than any single modality alone. A customer could look happy on camera but say something negative — or vice versa.
>
> **Real-world applications:** Brand auditing, content moderation, employee training assessment, customer research, call center analytics.

## 10 · Enhance the Customer Kiosk App

Everything we built above runs inside Workspace Notebook cells. But what if you wanted to share this as a **standalone application** that anyone in your organisation can open?

**What you're about to work with (the business view):** a self-serve **Coffee Menu Expert** kiosk — a simple web app a customer or counter-staff member can open, type a plain-English question ("Which drink has the least caffeine?", "What's the difference between a flat white and a latte?"), and get an instant answer drawn *only* from your roastery's own knowledge base. No barista lookup, no training required. It's the same parse → ground → answer pipeline you built above, wrapped in an interface anyone can use — which for the business means faster service, consistent answers, and menu expertise that scales without extra staff.

We've **pre-built** a working version (`coffee-kiosk/coffee_kiosk.py` in this lab's folder) that:
- Takes a customer's question via a text input
- Queries your `COFFEE_KNOWLEDGE_BASE` table
- Passes the question + knowledge base to `AI_COMPLETE` (pinned to `llama3.1-70b` for consistent results)
- Displays the answer

> :bulb: **Why pre-built?** CoCo *could* generate this from scratch — but the generation varies between runs (LLMs are non-deterministic). Starting from a known-good app lets you focus on the **iteration workflow** — the skill you'll actually use day-to-day — instead of debugging generation inconsistencies.

**Your task:** run the working app, then use CoCo to **enhance** it.

> 🔄 **Coming back to redo this section?** If you've already changed `coffee_kiosk.py` and want a clean slate, ask CoCo: *"Reset coffee_kiosk.py to its original version from coffee_kiosk_original.py"* — it will restore the starting state.

### Step 1 — Run the app

1. In the Workspace **file browser** (left panel), open `coffee-kiosk/` and click **`coffee_kiosk.py`**.
2. Click the **Run** button (top-right of the file editor). The app is already configured to use the shared `PLATPLUS_WH` warehouse, so there's nothing to set up — it just runs. The first launch may take 10–20 seconds while the container starts.
3. A split pane opens on the right with your running app. Type a question like **"What has the most caffeine?"** and press Enter.
4. The app queries your knowledge base, sends it to the LLM, and shows the answer.

> :rotating_light: **IMPORTANT — Don't see a Run button?** If you see an orange banner saying **"Convert to Streamlit app"** instead of a Run button, click that link, pick your warehouse (e.g. `PLATPLUS_WH`) when prompted, and click **Confirm**. The Run button will then appear. This happens if the `snowflake.yml` file was deleted from the `coffee-kiosk/` folder.

**What `coffee_kiosk.py` does**, end to end:
1. **Connects to Snowflake** — `st.connection("snowflake")` (no credentials; it runs inside Snowflake).
2. **Shows a text input** — the customer types a coffee question.
3. **Pulls the knowledge base** — `LISTAGG`s all rows of `COFFEE_KNOWLEDGE_BASE` into one text block.
4. **Calls `AI_COMPLETE`** — sends question + knowledge base to `llama3.1-70b` for a grounded answer.
5. **Displays the answer** with `st.markdown()`.

> :warning: **Your results may look different — that's normal.** LLMs are **non-deterministic**: same model, same prompt, slightly different wording each run. Focus on the *pattern*, not exact output.

> 💡 **Working across tabs:** from here you'll toggle between the **notebook** (instructions) and **`coffee_kiosk.py`** (the app). Click the tab names at the top to switch; the app keeps running in the background.

### Step 2 — Enhance with CoCo

Now use CoCo to add features. In the CoCo panel, type `/streamlit-in-workspaces` first to load the Streamlit skill, then paste one of these prompts.

> :rotating_light: **IMPORTANT:** These prompts **modify the existing `coffee_kiosk.py`** — they do NOT create a new app. Make sure you have `coffee_kiosk.py` open in its tab before pasting. CoCo will edit the file in place.

> :arrows_counterclockwise: **How to see your changes:** After each CoCo code change, restart the app to pick up the new code — either click the **3 dots (⋯)** on the app tab and select **Reboot**, or simply click **Run** again. (Don't see a 3-dots menu? Just click **Run**.)

🤖 **Enhancement 1 — Add a drink filter:**

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">In my coffee_kiosk.py app (in labs/09-unstructured-ai/coffee-kiosk/), add a sidebar with a multiselect that lets the user filter by specific drinks. Populate the options dynamically from the DRINK_ID column in the COFFEE_KNOWLEDGE_BASE table. When drinks are selected, only pass those drinks' content to the AI prompt; if none are selected, pass all drinks. Keep using llama3.1-70b for AI_COMPLETE.</pre>

🤖 **Enhancement 2 — Brand it & add a caffeine badge:**

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">In my coffee_kiosk.py app (in labs/09-unstructured-ai/coffee-kiosk/), brand it as "The Roastery Coffee Co.": show the logo roastery-logo.png (it's in this app folder) at the top and apply a warm coffee-brown color theme. Also, when the answer names a specific drink, detect which drink it is and show that drink's caffeine level as an st.metric badge beside the answer — derive the caffeine value from the COFFEE_KNOWLEDGE_BASE content.</pre>

> :bulb: **This is the real-world pattern.** You described what you wanted in plain English → CoCo edited the code → you ran it to verify. If something breaks, tell CoCo what you see ("the logo isn't showing", "the metric is blank for cold brew") and it fixes the code. No Stack Overflow — just a conversation.

> :rewind: **Want to start over?** Ask CoCo: *"Reset coffee_kiosk.py to its original version from coffee_kiosk_original.py"* — it will restore the clean starting state so you can try the enhancements again.

### Step 3 — What you just demonstrated

> **Pre-built app** → **CoCo enhanced it** → **you verified and iterated** → **the app is live with new features**

This is the real-world workflow: you rarely build from scratch. You start with something working and iterate with CoCo as your coding partner — describing what you want, reviewing the change, running to verify.

**What you can tell stakeholders:** "The kiosk answers customer questions about our full menu using our own documents. I added drink filtering and branding in minutes using CoCo."

**The production path (covered in the Admin persona notebook):**

| Concern | Who handles it | What they do |
|---------|---------------|-------------|
| Access control | Admin | Grant roles so the right people can reach the app |
| Compute & cost | Admin | Choose warehouse size, set auto-suspend, monitor usage |
| Scale the knowledge base | Engineer | Replace LISTAGG with Cortex Search for 100+ documents |
| Conversational interface | Engineer | Build a Cortex Agent for multi-turn Q&A via Snowflake CoWork |

> :bulb: **Your role as the analyst:** you proved the concept works and the AI gives grounded, accurate answers. That's the hard part — it validates the business case. The infrastructure decisions come after, and they're someone else's job.

---
### Check your understanding 🧠
A few questions on this lab's key themes. Run each quiz cell below, choose an answer, and review the feedback.

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q4")

---
### ❄️ Key Takeaways

**Core Functions:**
- **AI_PARSE_DOCUMENT** — Converts PDFs/images into queryable text (choose LAYOUT for structured docs, OCR for scans)
- **AI_EXTRACT** — Pulls named fields from free text into structured columns
- **AI_COMPLETE** — Generates answers grounded in your documents (RAG pattern)
- **AI_TRANSCRIBE** — Converts speech/audio to text for downstream analysis

**Key Patterns:**
- `DIRECTORY()` + `TO_FILE()` — iterate over staged files with SQL
- `AI_PARSE_DOCUMENT` → store → `AI_COMPLETE` with LISTAGG — the prototype RAG pattern
- Chain functions: parse → extract → classify → sentiment (each adds a layer of intelligence)
- CoCo as a coding partner: prompt → paste → run → iterate

**Best Practices:**
- Always spot-check parsed output against source documents before building on top
- LISTAGG works for small collections; Cortex Search is needed at scale (100+ docs)
- Test grounding: ask questions where you KNOW the answer and verify the AI cites your docs
- Multimodal (images, video, audio) uses the same `TO_FILE()` pattern — only the model changes
- Pre-compute and persist AI results rather than re-running expensive functions on every query

**What's Next:**
- The **Data Engineer** notebook builds the production version: Cortex Search Service + Agent
- The **Admin** notebook adds governance: PII redaction, content moderation, cost monitoring
- Together, all three personas create a complete, governed, customer-facing AI system

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.